# Basic EDA before the GDV analysis

This notebook is my first structured look at the StatsBomb data before building the final GDV figures.

I do not start directly with the final plots. First I check what data is available, which columns are useful, where values are missing and which parts of the data can realistically be used for my project. After that I focus only on the variables that I actually need for the final analysis.

Project idea: simple attacking patterns from the FIFA World Cup 2022 for amateur coaches.


## 1. Imports and folders

The notebook only uses standard data analysis packages and `statsbombpy`.  
The plots here are not meant to be final report figures. They are just used to understand the data.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from statsbombpy import sb
except ImportError as exc:
    raise ImportError("Please install statsbombpy first: pip install statsbombpy") from exc

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 10

# This works if the notebook is placed inside GDV/notebooks.
CURRENT_DIR = Path.cwd()
GDV_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
EDA_FIGURE_DIR = GDV_DIR / "figures" / "eda_overview"
EDA_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

EDA_FIGURE_DIR


## 2. Look at the available competitions

Before using the World Cup data, I first inspect the competition table.  
This is a small check, but it makes the analysis easier to reproduce because the competition and season IDs are taken from the data itself.


In [ ]:
competitions = sb.competitions()
print("Shape:", competitions.shape)
competitions.head(10)


In [ ]:
world_cup_2022 = competitions[
    competitions["competition_name"].str.contains("FIFA World Cup", case=False, na=False)
    & competitions["season_name"].astype(str).str.contains("2022", case=False, na=False)
].copy()

world_cup_2022[["competition_id", "season_id", "competition_name", "season_name", "country_name"]]


In [ ]:
if world_cup_2022.empty:
    raise ValueError("FIFA World Cup 2022 was not found in the StatsBomb competition table.")

COMPETITION_ID = int(world_cup_2022.iloc[0]["competition_id"])
SEASON_ID = int(world_cup_2022.iloc[0]["season_id"])

print("Competition ID:", COMPETITION_ID)
print("Season ID:", SEASON_ID)


## 3. Match table overview

Now I load all World Cup 2022 matches.  
At this point I mainly want to know how many matches are available and what match information can be used later.


In [ ]:
matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)

print("Number of matches:", len(matches))
print("Number of columns:", len(matches.columns))
matches.head()


In [ ]:
matches.info()


In [ ]:
useful_match_cols = [
    "match_id", "match_date", "competition_stage",
    "home_team", "away_team", "home_score", "away_score"
]

matches[[col for col in useful_match_cols if col in matches.columns]].head(12)


### What I can use from the match table

The match table is useful for context. I can use it to connect events with match names, stages and teams.

For the final GDV project I mainly need:
* `match_id` to load events
* team names for labels
* competition stage for possible context

The match table alone is not enough for the project because attacking patterns are stored in the event data.


## 4. Load a small event sample first

The full tournament has many events. For a first overview, I load only a few matches.  
This is enough to inspect columns, missing values and event types without making the notebook too slow.

Later, the final analysis notebook can process all matches.


In [ ]:
sample_matches = matches.head(3).copy()
sample_events_list = []

for _, match in sample_matches.iterrows():
    match_id = int(match["match_id"])
    print("Loading:", match_id, match["home_team"], "vs", match["away_team"])
    ev = sb.events(match_id=match_id)
    ev["match_id"] = match_id
    ev["match_label"] = f'{match["home_team"]} vs {match["away_team"]}'
    sample_events_list.append(ev)

events_sample = pd.concat(sample_events_list, ignore_index=True)

print("Sample event shape:", events_sample.shape)
events_sample.head()


## 5. What columns are available?

StatsBomb event data is wide. There are many columns, but not all are useful for my project.  
Here I look at the columns first and then check missing values.


In [ ]:
print("Number of columns:", len(events_sample.columns))
events_sample.columns.tolist()


In [ ]:
missing = (
    events_sample.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_share")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing.head(25)


In [ ]:
# Columns with low missing share are usually general event columns.
missing.tail(25)


### Short note about missing data

Many columns have missing values because they only belong to one event type.  
For example, shot columns are only filled for shots and pass columns are only filled for passes. This is normal in event data and not automatically a problem.

For the final project I should not simply drop columns only because they have many missing values. I first need to check whether the missing values are expected because of the event type.


## 6. Event types in the sample

This overview tells me which actions appear most often.  
It also shows whether the important event types for my project are present: passes, carries and shots.


In [ ]:
event_counts = events_sample["type"].value_counts().head(20).sort_values()

fig, ax = plt.subplots(figsize=(9, 7))
event_counts.plot(kind="barh", ax=ax)
ax.set_title("Most common event types in the sample")
ax.set_xlabel("Number of events")
ax.set_ylabel("Event type")

plt.tight_layout()
plt.savefig(EDA_FIGURE_DIR / "01_event_type_overview.png", dpi=150)
plt.show()


In [ ]:
important_types = ["Pass", "Carry", "Shot", "Ball Receipt*", "Duel", "Dribble"]
events_sample["type"].value_counts().reindex(important_types)


### What this means for the project

For my research question I do not need every event type.  
The most relevant events are:

* shots, because they are the attacking outcome
* goals, as a special shot outcome
* passes, because I count successful passes before shots
* carries, because they can be entries into the final third
* locations, because I need pitch zones and entry positions
* possessions, because I connect actions before a shot


## 7. Basic check of shots and goals

I now look only at shots.  
This is important because the final report uses shots, goals and expected goals.


In [ ]:
shots = events_sample[events_sample["type"].eq("Shot")].copy()

print("Number of shots in sample:", len(shots))
print("Shot columns that exist:")
[col for col in shots.columns if col.startswith("shot_")]


In [ ]:
shot_display_cols = [
    "match_label", "minute", "team", "player",
    "shot_outcome", "shot_statsbomb_xg", "location"
]
shots[[col for col in shot_display_cols if col in shots.columns]].head(12)


In [ ]:
if "shot_outcome" in shots.columns:
    outcome_counts = shots["shot_outcome"].value_counts().sort_values()

    fig, ax = plt.subplots(figsize=(8, 5))
    outcome_counts.plot(kind="barh", ax=ax)
    ax.set_title("Shot outcomes in the sample")
    ax.set_xlabel("Number of shots")
    ax.set_ylabel("Shot outcome")

    plt.tight_layout()
    plt.savefig(EDA_FIGURE_DIR / "02_shot_outcomes.png", dpi=150)
    plt.show()

outcome_counts if "shot_outcome" in shots.columns else "No shot_outcome column found"


In [ ]:
if "shot_statsbomb_xg" in shots.columns and not shots.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(shots["shot_statsbomb_xg"].dropna(), bins=12)
    ax.set_title("Distribution of xG values in the shot sample")
    ax.set_xlabel("StatsBomb xG")
    ax.set_ylabel("Number of shots")

    plt.tight_layout()
    plt.savefig(EDA_FIGURE_DIR / "03_xg_distribution.png", dpi=150)
    plt.show()

    shots["shot_statsbomb_xg"].describe()
else:
    print("No xG column found.")


### What I can use from shots

The shot data is useful for:

* counting shots
* identifying goals through `shot_outcome`
* using xG as a simple chance quality measure
* connecting shot outcomes to previous passes and final third entries

For the final report I use xG carefully. It is a chance quality estimate, not a guarantee that a shot should become a goal.


## 8. Basic check of passes

The final GDV report uses successful passes before shots.  
So I need to understand how passes and pass outcomes are represented.


In [ ]:
passes = events_sample[events_sample["type"].eq("Pass")].copy()

print("Number of passes in sample:", len(passes))
print("Pass columns that exist:")
[col for col in passes.columns if col.startswith("pass_")]


In [ ]:
pass_display_cols = [
    "match_label", "minute", "team", "player",
    "pass_outcome", "pass_end_location", "location"
]
passes[[col for col in pass_display_cols if col in passes.columns]].head(12)


In [ ]:
# In StatsBomb data, completed passes often have no pass_outcome value.
if "pass_outcome" in passes.columns:
    pass_status = passes["pass_outcome"].fillna("Complete").value_counts().sort_values()

    fig, ax = plt.subplots(figsize=(8, 5))
    pass_status.plot(kind="barh", ax=ax)
    ax.set_title("Pass outcomes in the sample")
    ax.set_xlabel("Number of passes")
    ax.set_ylabel("Pass outcome")

    plt.tight_layout()
    plt.savefig(EDA_FIGURE_DIR / "04_pass_outcomes.png", dpi=150)
    plt.show()

    pass_status
else:
    print("No pass_outcome column found.")


### What I can use from passes

Passes are useful for the final project because I can count completed passes before a shot in the same possession.  
I can also use pass start and end locations to check whether a pass entered the final third.

Important detail: in this dataset a missing `pass_outcome` usually means that the pass was completed. I therefore treat missing pass outcomes carefully instead of removing them.


## 9. Basic check of carries

Carries are also relevant because a player can move the ball into the final third without passing.  
For the final project I want to compare different entry methods, so carries should not be ignored.


In [ ]:
carries = events_sample[events_sample["type"].eq("Carry")].copy()

print("Number of carries in sample:", len(carries))
print("Carry columns that exist:")
[col for col in carries.columns if col.startswith("carry_")]


In [ ]:
carry_display_cols = [
    "match_label", "minute", "team", "player",
    "location", "carry_end_location"
]
carries[[col for col in carry_display_cols if col in carries.columns]].head(12)


### What I can use from carries

Carries are useful for identifying final third entries by dribbling or ball carrying.  
This makes the analysis more realistic than only looking at passes.


## 10. Check locations and final third entries

StatsBomb uses a 120 by 80 pitch coordinate system.  
For this project I define the final third simply as `x >= 80`.

This is a simplification, but it is easy to explain in the report.


In [ ]:
def get_location_value(value, index):
    if isinstance(value, list) and len(value) > index:
        return value[index]
    return np.nan

events_sample["x"] = events_sample["location"].apply(lambda value: get_location_value(value, 0))
events_sample["y"] = events_sample["location"].apply(lambda value: get_location_value(value, 1))

events_sample[["type", "location", "x", "y"]].head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(events_sample["x"].dropna(), bins=20)
ax.axvline(80, linestyle=":", linewidth=2, label="Final third starts at x = 80")
ax.set_title("Distribution of event x positions in the sample")
ax.set_xlabel("x position on StatsBomb pitch")
ax.set_ylabel("Number of events")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0)

plt.tight_layout()
plt.savefig(EDA_FIGURE_DIR / "05_event_x_positions.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
entry_events = events_sample[events_sample["type"].isin(["Pass", "Carry"])].copy()

entry_events["start_x"] = entry_events["location"].apply(lambda value: get_location_value(value, 0))
entry_events["start_y"] = entry_events["location"].apply(lambda value: get_location_value(value, 1))

entry_events["pass_end_x"] = np.nan
entry_events["pass_end_y"] = np.nan
entry_events["carry_end_x"] = np.nan
entry_events["carry_end_y"] = np.nan

if "pass_end_location" in entry_events.columns:
    entry_events["pass_end_x"] = entry_events["pass_end_location"].apply(lambda value: get_location_value(value, 0))
    entry_events["pass_end_y"] = entry_events["pass_end_location"].apply(lambda value: get_location_value(value, 1))

if "carry_end_location" in entry_events.columns:
    entry_events["carry_end_x"] = entry_events["carry_end_location"].apply(lambda value: get_location_value(value, 0))
    entry_events["carry_end_y"] = entry_events["carry_end_location"].apply(lambda value: get_location_value(value, 1))

entry_events["end_x"] = np.where(
    entry_events["type"].eq("Pass"),
    entry_events["pass_end_x"],
    entry_events["carry_end_x"]
)

final_third_entries = entry_events[
    (entry_events["start_x"] < 80)
    & (entry_events["end_x"] >= 80)
].copy()

print("Final third entries in the sample:", len(final_third_entries))
final_third_entries[["match_label", "minute", "team", "player", "type", "start_x", "end_x"]].head(12)


In [ ]:
if not final_third_entries.empty:
    entry_type_counts = final_third_entries["type"].value_counts().sort_values()

    fig, ax = plt.subplots(figsize=(7, 5))
    entry_type_counts.plot(kind="barh", ax=ax)
    ax.set_title("Final third entries by event type in the sample")
    ax.set_xlabel("Number of entries")
    ax.set_ylabel("Event type")

    plt.tight_layout()
    plt.savefig(EDA_FIGURE_DIR / "06_final_third_entry_types.png", dpi=150)
    plt.show()

    entry_type_counts
else:
    print("No final third entries found with this simple rule.")


### What I can use from location data

The location columns are useful for the final report because they allow simple spatial questions:

* where shots happen
* whether a pass or carry enters the final third
* from which simplified zones attacks start
* where a team creates chance quality

The limitation is that event data only stores ball actions. It does not show the full team shape or off ball runs.


## 11. Count passes before shots in the sample

This is one of the main ideas for the GDV report.  
Here I test the logic on the sample before using it on the full tournament data.


In [ ]:
def count_completed_passes_before_shots(match_events):
    rows = []

    if "possession" not in match_events.columns:
        return pd.DataFrame(rows)

    shot_rows = match_events[match_events["type"].eq("Shot")].copy()

    for _, shot in shot_rows.iterrows():
        previous_events = match_events[
            (match_events["match_id"] == shot["match_id"])
            & (match_events["possession"] == shot["possession"])
            & (match_events["index"] < shot["index"])
        ].copy()

        previous_passes = previous_events[previous_events["type"].eq("Pass")].copy()

        if "pass_outcome" in previous_passes.columns:
            completed_passes = previous_passes["pass_outcome"].isna().sum()
        else:
            completed_passes = len(previous_passes)

        rows.append(
            {
                "match_id": shot.get("match_id"),
                "match_label": shot.get("match_label"),
                "team": shot.get("team"),
                "player": shot.get("player"),
                "minute": shot.get("minute"),
                "shot_outcome": shot.get("shot_outcome"),
                "shot_xg": shot.get("shot_statsbomb_xg", np.nan),
                "completed_passes_before_shot": int(completed_passes),
            }
        )

    return pd.DataFrame(rows)

shot_sequence_sample = count_completed_passes_before_shots(events_sample)
shot_sequence_sample.head(12)


In [ ]:
if not shot_sequence_sample.empty:
    bins = [-1, 3, 6, 9, np.inf]
    labels = ["0 to 3", "4 to 6", "7 to 9", "10 or more"]

    shot_sequence_sample["pass_category"] = pd.cut(
        shot_sequence_sample["completed_passes_before_shot"],
        bins=bins,
        labels=labels
    )

    category_counts = shot_sequence_sample["pass_category"].value_counts().reindex(labels)

    fig, ax = plt.subplots(figsize=(8, 5))
    category_counts.plot(kind="bar", ax=ax)
    ax.set_title("Pass categories before shots in the sample")
    ax.set_xlabel("Completed passes before the shot")
    ax.set_ylabel("Number of shots")
    ax.tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.savefig(EDA_FIGURE_DIR / "07_pass_categories_before_shots.png", dpi=150)
    plt.show()

    category_counts
else:
    print("No shots found in the sample.")


### Why I keep this idea for the final analysis

This check shows that the data can be used to connect shots with previous possession events.  
For the final GDV figures I use this logic on the full tournament data, not only on the sample.

The pass categories are simple enough for the target audience. They are not perfect tactical labels, but they are understandable for a first visual analysis.


## 12. What I do not use in the final project

Not every column is useful for this GDV submission.  
I decided not to use some parts because they would make the project too broad or because the data is incomplete for my question.

Examples:

* player tracking is not available, so I cannot analyse off ball movement
* tactical instructions are not available, so I cannot know the coach's plan
* many event specific columns are only filled for one event type
* goalkeeper and defensive event details are interesting, but not central for my attacking pattern question
* exact formation context is only used carefully, because formations alone do not explain attacking quality


## 13. Final variables used for the GDV project

After this EDA, I focus on these parts of the data:

| Data part | Why I use it |
| --- | --- |
| shots | final attacking outcome |
| goals | successful shot outcome |
| xG | simple chance quality measure |
| completed passes before shots | build up length before an attempt |
| pass and carry locations | entries into the final third |
| possession ID | links actions before a shot |
| team names | team comparisons and case studies |
| match context | basic metadata and labels |

This is enough for a focused GDV project. It also keeps the analysis explainable for amateur coaches instead of turning it into a complex tactical model.


## 14. Short conclusion from the EDA

The data is useful for my project, but it has to be interpreted carefully.

The strongest part of the data is that it gives detailed ball events with locations. This makes it possible to study passes, carries, shots, goals and entries into the final third.

The weakest part is that it does not show the whole tactical picture. I cannot see all player runs, defensive pressure or coaching instructions. Because of that, the final report stays descriptive. I use the visualizations to show possible attacking patterns, not to prove that one method is always better.
